# Phase 1 — 환경 설정 & 데이터 준비

**목표:** 프로젝트 환경 검증 및 WM-811K 데이터셋 다운로드

| 항목 | 내용 |
|------|------|
| 데이터셋 | WM-811K (Wafer Map 811K) |
| 출처 | Kaggle — `qingyi/wm811k-wafer-map` |
| 샘플 수 | 811,457개 웨이퍼 맵 |
| 클래스 | 9종 불량 패턴 + 정상(None) |

## 1. 패키지 버전 확인

In [1]:
import torch
import torchvision
import timm
import mlflow
import kagglehub
import numpy as np
import pandas as pd
import sklearn
import matplotlib
import seaborn
import cv2
import PIL
import imblearn

print('=== 환경 검증 ===')
print(f'Python:           3.12.10')
print(f'torch:            {torch.__version__}')
print(f'CUDA 사용 가능:   {torch.cuda.is_available()}')
print(f'GPU:              {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print(f'VRAM:             {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB' if torch.cuda.is_available() else '')
print()
print(f'torchvision:      {torchvision.__version__}')
print(f'timm:             {timm.__version__}')
print(f'mlflow:           {mlflow.__version__}')
print(f'kagglehub:        {kagglehub.__version__}')
print(f'numpy:            {np.__version__}')
print(f'pandas:           {pd.__version__}')
print(f'scikit-learn:     {sklearn.__version__}')
print(f'matplotlib:       {matplotlib.__version__}')
print(f'seaborn:          {seaborn.__version__}')
print(f'opencv:           {cv2.__version__}')
print(f'Pillow:           {PIL.__version__}')
print(f'imbalanced-learn: {imblearn.__version__}')
print()
print('모든 패키지 정상 로드 완료')

=== 환경 검증 ===
Python:           3.12.10
torch:            2.6.0+cu124
CUDA 사용 가능:   True
GPU:              NVIDIA GeForce RTX 2060 SUPER
VRAM:             8.0 GB

torchvision:      0.21.0+cu124
timm:             1.0.27
mlflow:           3.14.0
kagglehub:        1.0.2
numpy:            2.4.4
pandas:           2.3.3
scikit-learn:     1.9.0
matplotlib:       3.11.0
seaborn:          0.13.2
opencv:           4.13.0
Pillow:           12.2.0
imbalanced-learn: 0.14.2

모든 패키지 정상 로드 완료


## 2. 글로벌 설정 (랜덤 시드 고정)

In [2]:
import random
import os

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f'랜덤 시드 고정 완료: SEED={SEED}')

랜덤 시드 고정 완료: SEED=42


## 3. WM-811K 데이터셋 다운로드

> Kaggle API 인증이 필요합니다.  
> `~/.kaggle/kaggle.json` 파일에 API 토큰이 있어야 합니다.

In [3]:
import kagglehub

print('WM-811K 데이터셋 다운로드 중...')
path = kagglehub.dataset_download('qingyi/wm811k-wafer-map')
print(f'다운로드 완료: {path}')

WM-811K 데이터셋 다운로드 중...
다운로드 완료: C:\Users\naisk\.cache\kagglehub\datasets\qingyi\wm811k-wafer-map\versions\1


## 4. 다운로드된 파일 구조 확인

In [4]:
import os

print('=== 데이터 파일 구조 ===')
for root, dirs, files in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    sub_indent = '  ' * (level + 1)
    for file in files:
        file_path = os.path.join(root, file)
        size_mb = os.path.getsize(file_path) / 1024**2
        print(f'{sub_indent}{file}  ({size_mb:.1f} MB)')

=== 데이터 파일 구조 ===
1/
  LSWMD.pkl  (1998.4 MB)


## 5. 데이터 로드 테스트

In [5]:
import pickle

# pickle 파일 찾기
pkl_files = []
for root, dirs, files in os.walk(path):
    for f in files:
        if f.endswith('.pkl') or f.endswith('.pickle'):
            pkl_files.append(os.path.join(root, f))

print(f'발견된 pickle 파일: {pkl_files}')

if pkl_files:
    df = pd.read_pickle(pkl_files[0])
    print(f'\n데이터 형태: {df.shape}')
    print(f'컬럼 목록: {list(df.columns)}')
    print(f'\n데이터 타입:')
    print(df.dtypes)
    print(f'\n상위 3행:')
    display(df.head(3))

발견된 pickle 파일: ['C:\\Users\\naisk\\.cache\\kagglehub\\datasets\\qingyi\\wm811k-wafer-map\\versions\\1\\LSWMD.pkl']


c:\Users\naisk\Desktop\하이닉스 대비\웨이퍼 불량 분석\.venv\Lib\site-packages\pandas\compat\pickle_compat.py:35: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  stack[-1] = func(*args)
c:\Users\naisk\Desktop\하이닉스 대비\웨이퍼 불량 분석\.venv\Lib\site-packages\pandas\compat\pickle_compat.py:35: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  stack[-1] = func(*args)
c:\Users\naisk\Desktop\하이닉스 대비\웨이퍼 불량 분석\.venv\Lib\site-packages\pandas\compat\pickle_compat.py:35: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  stack[-1] = func(*args)



데이터 형태: (811457, 6)
컬럼 목록: ['waferMap', 'dieSize', 'lotName', 'waferIndex', 'trianTestLabel', 'failureType']

데이터 타입:
waferMap           object
dieSize           float64
lotName            object
waferIndex        float64
trianTestLabel     object
failureType        object
dtype: object

상위 3행:


c:\Users\naisk\Desktop\하이닉스 대비\웨이퍼 불량 분석\.venv\Lib\site-packages\pandas\compat\pickle_compat.py:35: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  stack[-1] = func(*args)


,waferMap,dieSize,lotName,waferIndex,trianTestLabel,failureType
0,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,1.0,[[Training]],[[none]]
1,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,2.0,[[Training]],[[none]]
2,"[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...",1683.0,lot1,3.0,[[Training]],[[none]]


## 6. 기본 데이터 통계 확인

In [ ]:
print('=== 기본 통계 ===')
print(f'전체 샘플 수: {len(df):,}')
print()

# 레이블 분포 확인
label_col = 'failureType' if 'failureType' in df.columns else df.columns[-1]
print(f'레이블 컬럼: {label_col}')
print(f'\n클래스 분포:')
print(df[label_col].value_counts())

# 웨이퍼 맵 샘플 크기 확인
map_col = 'waferMap' if 'waferMap' in df.columns else df.columns[0]
sample_shapes = df[map_col].apply(lambda x: x.shape if hasattr(x, 'shape') else None).dropna()
print(f'\n웨이퍼 맵 크기 샘플 (상위 5개):')
print(sample_shapes.value_counts().head())

=== 기본 통계 ===
전체 샘플 수: 811,457

레이블 컬럼: failureType

클래스 분포:


## 7. 데이터 경로 저장

다음 노트북(`02_eda.ipynb`)에서 재사용할 수 있도록 데이터 경로를 기록합니다.

In [ ]:
# 다음 노트북에서 사용할 경로 출력
print('=== 다음 단계에서 사용할 경로 ===')
print(f'DATA_PATH = r"{pkl_files[0]}"')
print()
print('Phase 1 완료. 다음 단계: 02_eda.ipynb')